In [1]:
import os
import json
import base64
import requests
from openai import OpenAI

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

# ========== 环境参数（优先从 .env 读取，否则使用默认值） ==========
HOST_IP = os.getenv("HOST_IP", "192.168.1.13")

LLM_PORT = int(os.getenv("LLM_PORT", "10001"))
VLM_PORT = int(os.getenv("VLM_PORT", "10002"))
ASR_PORT = int(os.getenv("ASR_PORT", "10003"))
EMBEDDING_PORT = int(os.getenv("EMBEDDING_PORT", "10004"))
RERANKER_PORT = int(os.getenv("RERANKER_PORT", "10005"))

API_KEY = os.getenv("API_KEY", "xtsk")


def openai_client(port: int) -> OpenAI:
    """根据端口号创建指向本地服务的 OpenAI 客户端。"""
    return OpenAI(
        base_url=f"http://{HOST_IP}:{port}/v1",
        api_key=API_KEY,
    )


def auth_headers() -> dict:
    """构造带 Bearer Token 的请求头（仅 reranker 等无 SDK 原生支持的接口使用）。"""
    return {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}",
    }

In [2]:
# ========== LLM 文本对话（OpenAI SDK） ==========

client = openai_client(LLM_PORT)

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "用一句话解释什么是向量检索"},
]

# --- 非流式 ---
print("=== LLM 非流式 ===")
resp = client.chat.completions.create(
    model="LLM",
    messages=messages,
    temperature=0.2,
)
print(resp.choices[0].message.content)

# --- 流式 ---
print("\n=== LLM 流式 ===")
stream = client.chat.completions.create(
    model="LLM",
    messages=messages,
    temperature=0.2,
    stream=True,
)
# 逐块读取增量内容
for event in stream:
    delta = event.choices[0].delta
    if delta and delta.content:
        print(delta.content, end="", flush=True)
print()

=== LLM 非流式 ===
向量检索是一种通过计算和比较高维向量之间的相似度（如余弦相似度或欧氏距离），从大规模数据集中快速找出与查询向量最相似项的技术。

=== LLM 流式 ===
向量检索是一种通过计算和比较高维向量之间的相似度（如余弦相似度或欧氏距离），从海量数据中快速找出与查询向量最相似项的技术。


In [ ]:
# ========== VLM 图文理解（OpenAI SDK） ==========

local_image = "test.jpg"  # 请确保文件存在
if not os.path.exists(local_image):
    raise FileNotFoundError(f"image file not found: {local_image}")

# 读取图片并 base64 编码
with open(local_image, "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode("utf-8")

client = openai_client(VLM_PORT)

# VLM 消息体：文本 + base64 图片
messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "请描述图片里最主要的内容。"},
            {
                "type": "image_url",
                # 通过 data URI 传递 base64 编码的图片
                "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"},
            },
        ],
    }
]

# --- 非流式 ---
print("=== VLM 非流式 ===")
resp = client.chat.completions.create(
    model="VLM",
    messages=messages,
    temperature=0.2,
)
print(resp.choices[0].message.content)

# --- 流式 ---
print("\n=== VLM 流式 ===")
stream = client.chat.completions.create(
    model="VLM",
    messages=messages,
    temperature=0.2,
    stream=True,
)
# 逐块读取增量内容
for event in stream:
    delta = event.choices[0].delta
    if delta and delta.content:
        print(delta.content, end="", flush=True)
print()

In [ ]:
# ========== ASR 语音识别（OpenAI SDK） ==========

local_audio = "di-test-sr-1.wav"  # 请确保文件存在
if not os.path.exists(local_audio):
    raise FileNotFoundError(f"audio file not found: {local_audio}")

# 读取音频并 base64 编码
with open(local_audio, "rb") as f:
    audio_b64 = base64.b64encode(f.read()).decode("utf-8")

client = openai_client(ASR_PORT)

# ASR 通过 audio_url 类型传递 base64 音频数据
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "audio_url",
                "audio_url": {"url": f"data:audio/wav;base64,{audio_b64}"},
            }
        ],
    }
]

# --- 非流式 ---
print("=== ASR 非流式 ===")
resp = client.chat.completions.create(
    model="ASR",
    messages=messages,
)
print(resp.choices[0].message.content)

# --- 流式 ---
print("\n=== ASR 流式 ===")
stream = client.chat.completions.create(
    model="ASR",
    messages=messages,
    stream=True,
)
for event in stream:
    delta = event.choices[0].delta
    if delta and delta.content:
        print(delta.content, end="", flush=True)
print()

In [ ]:
# ========== Embedding 向量化（OpenAI SDK） ==========

client = openai_client(EMBEDDING_PORT)

texts = [
    "向量数据库适合存什么？",
    "embedding 有什么用？",
]

# embeddings.create 返回每条文本对应的向量
resp = client.embeddings.create(
    model="EMBEDDING",
    input=texts,
)
print("dim =", len(resp.data[0].embedding))
print("first 20 =", resp.data[0].embedding[:20])

In [ ]:
# ========== Reranker 重排序（requests） ==========
# 注意：OpenAI SDK 没有原生 rerank 方法，因此这里使用 requests 直接调用 /rerank 端点

url = f"http://{HOST_IP}:{RERANKER_PORT}/rerank"

payload = {
    "model": "RERANKER",
    "query": "如何办理退税？",
    "documents": [
        "你可以在 APP 里提交申请。",
        "退税需要满足一定条件，并准备相关材料。",
        "今天天气不错。",
    ],
    "top_n": 2,
}

try:
    resp = requests.post(url, headers=auth_headers(), json=payload, timeout=300)
    resp.raise_for_status()
    print(resp.json())
except Exception as e:
    print("reranker 调用失败:", e)